In [5]:
%conda install -c conda-forge m2w64-toolchain -y

3 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - conda-forge
 - defaults
Platform: win-64
Solving environment: done

## Package Plan ##

  environment location: c:\Users\Asus\anaconda3\envs\ai_proje

  added / updated specs:
    - m2w64-toolchain


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.6.17  |       h4c7d964_0         126 KB  conda-forge
    certifi-2026.6.17          |     pyhd8ed1ab_0         131 KB  conda-forge
    m2w64-binutils-2.25.1      |                5        44.3 MB  conda-forge
    m2w64-bzip2-1.0.6          |                6         102 KB  conda-forge
    m2w64-crt-git-5.0.0.4636.2595836|                2         3.4 MB  conda-forge
    m2w64-gcc-5.3.0            |                6        40.8 MB  conda-forge
    m2w64-gcc-ada-5.3.0        |                6        33.3 MB  conda-forge
    m2w64-gcc-fortran-5.3.0    



==> WARNING: A newer version of conda exists. <==
    current version: 25.5.1
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c defaults conda




In [1]:
! pip install pymc bambi arviz

INFO: pip is looking at multiple versions of pymc to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/535.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/535.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/535.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/535.5 kB ? eta -:--:--
   ------------------- -------------------- 262.1/535.5 kB ? eta -:--:--
   ------------------- -------------------- 262.1/535.5 kB ? eta -:--:--
   ------------------- -------------------- 262.1/535.5 kB ? eta -:--:--
   ------------------- -------------------- 262.1/535.5 kB ? eta -:--:--
   ---------------------------------------- 535.5/535.5 kB 357.4 kB/s  0:00:01
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_theme(style="whitegrid", palette="muted")

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, Matern, RationalQuadratic
from sklearn.metrics import mean_absolute_error, mean_squared_error

import scipy.stats as stats

import pymc as pm
import bambi as bmb
import arviz as az

print(f"PyMC Versiyonu: {pm.__version__} | Bambi Versiyonu: {bmb.__version__} | ArviZ Versiyonu: {az.__version__}")

PyMC Versiyonu: 5.23.0 | Bambi Versiyonu: 0.15.0 | ArviZ Versiyonu: 0.23.4


In [2]:
df = pd.read_csv("C:/Users/Asus/montesinho-fire-risk-prediction/data/raw/forestfires.csv")

print("Veri Boyutu:", df.shape)
df.describe().T

df.head(10)

Veri Boyutu: (517, 13)


,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.0
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.0
5,8,6,aug,sun,92.3,85.3,488.0,14.7,22.2,29,5.4,0.0,0.0
6,8,6,aug,mon,92.3,88.9,495.6,8.5,24.1,27,3.1,0.0,0.0
7,8,6,aug,mon,91.5,145.4,608.2,10.7,8.0,86,2.2,0.0,0.0
8,8,6,sep,tue,91.0,129.5,692.6,7.0,13.1,63,5.4,0.0,0.0
9,7,5,sep,sat,92.5,88.0,698.6,7.1,22.8,40,4.0,0.0,0.0


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df['month'] = df['month'].map({'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6, 'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12})
df['day'] = df['day'].map({'mon': 1, 'tue': 2, 'wed': 3, 'thu': 4, 'fri': 5, 'sat': 6, 'sun': 7})

X = df.drop(columns=['area'])
y = df['area'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

if "probabilistic_results" not in globals():
    probabilistic_results = []

def evaluate_probabilistic_model(y_true, y_pred_mean, y_pred_lower, y_pred_upper, stage_name, interval_level=90):
    mad_val = np.median(np.abs(y_true - y_pred_mean))
    mae_val = mean_absolute_error(y_true, y_pred_mean)
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred_mean))
    covered = (y_true >= y_pred_lower) & (y_true <= y_pred_upper)
    picp_val = np.mean(covered) * 100.0
    mpiw_val = np.mean(y_pred_upper - y_pred_lower)
    
    existing_names = [r.get("Deney Aşaması") for r in probabilistic_results]
    entry_dict = {
        "Deney Aşaması": stage_name,
        "MAD (ha)": mad_val,
        "MAE (ha)": mae_val,
        "RMSE (ha)": rmse_val,
        f"%{interval_level} PICP (Kapsama)": picp_val,
        f"%{interval_level} MPIW (Genişlik)": mpiw_val
    }
    
    if stage_name not in existing_names:
        probabilistic_results.append(entry_dict)
    else:
        for idx_item, item_row in enumerate(probabilistic_results):
            if item_row.get("Deney Aşaması") == stage_name:
                probabilistic_results[idx_item] = entry_dict
                
    print("=========================================================================================")
    print(f"[DEFTER 9] OLASILIKSAL MODEL - {stage_name.upper()} SONUÇLARI")
    print("=========================================================================================")
    print(f"-> MAD: {mad_val:.4f} ha | MAE: {mae_val:.4f} ha | RMSE: {rmse_val:.4f} ha")
    print(f"-> %{interval_level} PICP (Kapsama): %{picp_val:.2f} | MPIW (Ort. Genişlik): {mpiw_val:.2f} ha")
    print("-----------------------------------------------------------------------------------------")
    
    df_prob = pd.DataFrame(probabilistic_results)
    styled_table = (
        df_prob.style
        .background_gradient(subset=["MAD (ha)", "MAE (ha)"], cmap="Purples")
        .background_gradient(subset=[f"%{interval_level} PICP (Kapsama)"], cmap="YlGn", vmin=interval_level-10, vmax=100)
        .format({
            "MAD (ha)": "{:.4f}",
            "MAE (ha)": "{:.4f}",
            "RMSE (ha)": "{:.4f}",
            f"%{interval_level} PICP (Kapsama)": "%{:.2f}",
            f"%{interval_level} MPIW (Genişlik)": "{:.2f}"
        })
        .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
        .set_properties(subset=["Deney Aşaması"], **{'font-weight': 'bold', 'color': '#4a148c'})
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#4a148c'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
        ])
    )
    display(styled_table)
    return mad_val, mae_val, rmse_val, picp_val, mpiw_val

##  Gaussian Process Regression (GPR) Deneyleri:

### Deney 1:

In [11]:
kernel_gpr_normal = 1.0 * Matern(length_scale=1.0, nu=1.5) + WhiteKernel(noise_level=1.0)
model_gpr_normal = GaussianProcessRegressor(kernel=kernel_gpr_normal, n_restarts_optimizer=10, normalize_y=True, random_state=42)
model_gpr_normal.fit(X_train_scaled, y_train)

y_pred_mean_normal, y_pred_std_normal = model_gpr_normal.predict(X_test_scaled, return_std=True)

z_score_90 = stats.norm.ppf(0.95)

y_pred_mean_clipped = np.clip(y_pred_mean_normal, 0.0, None)
y_pred_lower_normal = np.clip(y_pred_mean_normal - z_score_90 * y_pred_std_normal, 0.0, None)
y_pred_upper_normal = y_pred_mean_normal + z_score_90 * y_pred_std_normal

_ = evaluate_probabilistic_model(y_test, y_pred_mean_clipped, y_pred_lower_normal, y_pred_upper_normal, "Deney 1A - GPR (Normal Veri)")

[DEFTER 9] OLASILIKSAL MODEL - DENEY 1A - GPR (NORMAL VERI) SONUÇLARI
-> MAD: 11.1321 ha | MAE: 24.2301 ha | RMSE: 108.9081 ha
-> %90 PICP (Kapsama): %97.12 | MPIW (Ort. Genişlik): 84.95 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,MAD (ha),MAE (ha),RMSE (ha),%90 PICP (Kapsama),%90 MPIW (Genişlik)
0,Deney 1A - GPR (Normal Veri),11.1321,24.2301,108.9081,%97.12,84.95


### Deney 2:

In [13]:
print("Deney 3 (GPR - Log Veri) Eğitimi Başlıyor...")

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

kernel_gpr_log = 1.0 * Matern(length_scale=1.0, nu=1.5) + WhiteKernel(noise_level=1.0)
model_gpr_log = GaussianProcessRegressor(kernel=kernel_gpr_log, n_restarts_optimizer=10, normalize_y=True, random_state=42)
model_gpr_log.fit(X_train_scaled, y_train_log)

y_pred_mean_log, y_pred_std_log = model_gpr_log.predict(X_test_scaled, return_std=True)

y_pred_mean_log_reverted = np.expm1(y_pred_mean_log)

y_pred_lower_log = y_pred_mean_log - z_score_90 * y_pred_std_log
y_pred_upper_log = y_pred_mean_log + z_score_90 * y_pred_std_log

y_pred_lower_log_reverted = np.expm1(y_pred_lower_log)
y_pred_upper_log_reverted = np.expm1(y_pred_upper_log)

_ = evaluate_probabilistic_model(y_test, y_pred_mean_log_reverted, y_pred_lower_log_reverted, y_pred_upper_log_reverted, "Deney 3 - GPR (Log Veri)")

Deney 3 (GPR - Log Veri) Eğitimi Başlıyor...
[DEFTER 9] OLASILIKSAL MODEL - DENEY 3 - GPR (LOG VERI) SONUÇLARI
-> MAD: 2.0150 ha | MAE: 19.7966 ha | RMSE: 109.9946 ha
-> %90 PICP (Kapsama): %88.46 | MPIW (Ort. Genişlik): 28.49 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,MAD (ha),MAE (ha),RMSE (ha),%90 PICP (Kapsama),%90 MPIW (Genişlik)
0,Deney 1A - GPR (Normal Veri),11.1321,24.2301,108.9081,%97.12,84.95
1,Deney 1A - GPR (Normal Veri - Optuna HPO),11.1321,24.2301,108.9081,%97.12,84.91
2,Deney 3 - GPR (Log Veri),2.0150,19.7966,109.9946,%88.46,28.49


## BART (Bayesian Additive Regression Trees) Deneyleri:

In [19]:
%conda install -c conda-forge pymc-bart -y

3 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: win-64
Solving environment: done

## Package Plan ##

  environment location: c:\Users\Asus\anaconda3\envs\ai_proje

  added / updated specs:
    - pymc-bart


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    arviz-0.23.4               |     pyhcf101f3_0         1.4 MB  conda-forge
    cachetools-7.1.4           |     pyhd8ed1ab_0          21 KB  conda-forge
    cloudpickle-3.1.2          |     pyhcf101f3_1          27 KB  conda-forge
    cons-0.4.7                 |     pyhd8ed1ab_2          14 KB  conda-forge
    etuples-0.3.10             |     pyhd8ed1ab_1          18 KB  conda-forge
    filelock-3.30.3            |     pyhd8ed1ab_0          73 KB  conda-forge
    getopt-win32-0.1           |       h6a83c73_3          26 KB  conda-forge
    graphviz-9.0.0             |       h51cb2cd_1         1.1 MB  c



==> WARNING: A newer version of conda exists. <==
    current version: 25.5.1
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c defaults conda




In [6]:
import pytensor
import pymc as pm
import pymc_bart as pmb
import numpy as np

pytensor.config.cxx = ""

print("Deney 2A (BART - Normal Veri) Eğitimi Başlıyor (Saf Python Motoru ile)...")

with pm.Model() as model_bart_normal:
    X_data = pm.Data("X", X_train_scaled)
    y_data = pm.Data("y", y_train)
    
    mu = pmb.BART("mu", X_data, y_data, m=20)
    sigma = pm.HalfNormal("sigma", sigma=50)
    
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_data)
    
    idata_bart_normal = pm.sample(draws=250, tune=250, chains=2, random_seed=42, cores=1, progressbar=True, compute_convergence_checks=False)
    
    pm.set_data({"X": X_test_scaled, "y": np.zeros(len(X_test_scaled))})
    pm.sample_posterior_predictive(idata_bart_normal, extend_inferencedata=True, predictions=True, random_seed=42, progressbar=False)

posterior_samples_bart_normal = idata_bart_normal.predictions["y_obs"].values.reshape(-1, len(y_test))

y_pred_mean_bart = np.clip(np.mean(posterior_samples_bart_normal, axis=0), 0.0, None)
y_pred_lower_bart = np.clip(np.percentile(posterior_samples_bart_normal, 5, axis=0), 0.0, None)
y_pred_upper_bart = np.percentile(posterior_samples_bart_normal, 95, axis=0)

_ = evaluate_probabilistic_model(y_test, y_pred_mean_bart, y_pred_lower_bart, y_pred_upper_bart, "Deney 2A - BART (Normal Veri)")

Deney 2A (BART - Normal Veri) Eğitimi Başlıyor (Saf Python Motoru ile)...


Sequential sampling (2 chains in 1 job)
CompoundStep
>PGBART: [mu]
>NUTS: [sigma]


Output()

Sampling 2 chains for 250 tune and 250 draw iterations (500 + 500 draws total) took 486 seconds.
Sampling: [mu, y_obs]


[DEFTER 9] OLASILIKSAL MODEL - DENEY 2A - BART (NORMAL VERI) SONUÇLARI
-> MAD: 9.2793 ha | MAE: 24.1368 ha | RMSE: 108.5605 ha
-> %90 PICP (Kapsama): %97.12 | MPIW (Ort. Genişlik): 84.41 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,MAD (ha),MAE (ha),RMSE (ha),%90 PICP (Kapsama),%90 MPIW (Genişlik)
0,Deney 2A - BART (Normal Veri),9.2793,24.1368,108.5605,%97.12,84.41


In [11]:
import pytensor
import pymc as pm
import pymc_bart as pmb
import numpy as np

pytensor.config.cxx = ""

print("Deney 2B (BART - Log Veri) Eğitimi Başlıyor...")

X_train_safe = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
X_test_safe = np.nan_to_num(X_test_scaled, nan=0.0, posinf=0.0, neginf=0.0)
y_train_log_safe = np.nan_to_num(np.log1p(y_train), nan=0.0, posinf=0.0, neginf=0.0)

with pm.Model() as model_bart_log:
    X_data_log = pm.Data("X_log", X_train_safe)
    y_data_log = pm.Data("y_log", y_train_log_safe)
    
    mu = pmb.BART("mu", X_data_log, y_data_log, m=20)
    sigma = pm.HalfNormal("sigma", sigma=10)
    
    y_obs = pm.Normal("y_obs_log", mu=mu, sigma=sigma, observed=y_data_log)
    
    idata_bart_log = pm.sample(draws=250, tune=250, chains=2, random_seed=42, cores=1, progressbar=True, compute_convergence_checks=False)
    
    pm.set_data({"X_log": X_test_safe, "y_log": np.zeros(len(X_test_safe))})
    pm.sample_posterior_predictive(idata_bart_log, extend_inferencedata=True, predictions=True, random_seed=42, progressbar=False)

posterior_samples_bart_log = idata_bart_log.predictions["y_obs_log"].values.reshape(-1, len(y_test))
posterior_samples_bart_reverted = np.expm1(posterior_samples_bart_log)

y_pred_mean_bart_log = np.mean(posterior_samples_bart_reverted, axis=0)
y_pred_lower_bart_log = np.percentile(posterior_samples_bart_reverted, 5, axis=0)
y_pred_upper_bart_log = np.percentile(posterior_samples_bart_reverted, 95, axis=0)

_ = evaluate_probabilistic_model(y_test, y_pred_mean_bart_log, y_pred_lower_bart_log, y_pred_upper_bart_log, "Deney 2B - BART (Log Veri)")

Deney 2B (BART - Log Veri) Eğitimi Başlıyor...


Sequential sampling (2 chains in 1 job)
CompoundStep
>PGBART: [mu]
>NUTS: [sigma]


Output()

Sampling 2 chains for 250 tune and 250 draw iterations (500 + 500 draws total) took 398 seconds.
Sampling: [mu, y_obs_log]


[DEFTER 9] OLASILIKSAL MODEL - DENEY 2B - BART (LOG VERI) SONUÇLARI
-> MAD: 6.4702 ha | MAE: 21.6842 ha | RMSE: 109.2254 ha
-> %90 PICP (Kapsama): %89.42 | MPIW (Ort. Genişlik): 27.68 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,MAD (ha),MAE (ha),RMSE (ha),%90 PICP (Kapsama),%90 MPIW (Genişlik)
0,Deney 2A - BART (Normal Veri),9.2793,24.1368,108.5605,%97.12,84.41
1,Deney 2B - BART (Log Veri),6.4702,21.6842,109.2254,%89.42,27.68


## HİYERARŞİK BAYES Deneyleri:

In [19]:
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import OneHotEncoder
import scipy.stats as stats
import numpy as np

print("Deney 3 - Bayesian Ridge (Matematiksel Hiyerarşi) Eğitimi Başlıyor...")

y_train_log = np.log1p(y_train)

# 1. ZIRH: Gizli zehirlenmeleri (NaN/Inf) anında sıfıra eziyoruz!
X_train_safe = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
X_test_safe = np.nan_to_num(X_test_scaled, nan=0.0, posinf=0.0, neginf=0.0)

# 2. HİYERARŞİ: X ve Y grid koordinatlarını Kategorik (One-Hot) olarak modele ekliyoruz
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_grid_train = encoder.fit_transform(X_train[['X', 'Y']])
X_grid_test = encoder.transform(X_test[['X', 'Y']])

# Ölçeklenmiş ana verilerle grid (uzaysal) verilerini birleştiriyoruz
X_train_hier = np.hstack((X_train_safe, X_grid_train))
X_test_hier = np.hstack((X_test_safe, X_grid_test))

# Bayesian Ridge (Hiyerarşik Kısmi Havuzlama matematiğinin ta kendisidir!)
model_bayesian_ridge = BayesianRidge(max_iter=1000)
model_bayesian_ridge.fit(X_train_hier, y_train_log)

# Hem ortalama tahmini (mean) hem de belirsizliği (std) aynı anda alıyoruz!
y_pred_log_mean, y_pred_log_std = model_bayesian_ridge.predict(X_test_hier, return_std=True)

# %90 Güven Aralığı (Z-score: 1.645)
z_score_90 = stats.norm.ppf(0.95)

y_pred_lower_log = y_pred_log_mean - z_score_90 * y_pred_log_std
y_pred_upper_log = y_pred_log_mean + z_score_90 * y_pred_log_std

# Log uzayından gerçek (Hektar) uzayına geri dönüş
y_pred_mean_hier_reverted = np.expm1(y_pred_log_mean)
y_pred_lower_hier_reverted = np.clip(np.expm1(y_pred_lower_log), 0.0, None)
y_pred_upper_hier_reverted = np.expm1(y_pred_upper_log)

_ = evaluate_probabilistic_model(y_test, y_pred_mean_hier_reverted, y_pred_lower_hier_reverted, y_pred_upper_hier_reverted, "Deney 3 - Bayesian Ridge (Matematiksel Hiyerarşi)")

Deney 3 - Bayesian Ridge (Matematiksel Hiyerarşi) Eğitimi Başlıyor...
[DEFTER 9] OLASILIKSAL MODEL - DENEY 3 - BAYESIAN RIDGE (MATEMATIKSEL HIYERARŞI) SONUÇLARI
-> MAD: 2.0318 ha | MAE: 19.8132 ha | RMSE: 109.9892 ha
-> %90 PICP (Kapsama): %89.42 | MPIW (Ort. Genişlik): 27.88 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,MAD (ha),MAE (ha),RMSE (ha),%90 PICP (Kapsama),%90 MPIW (Genişlik)
0,Deney 2A - BART (Normal Veri),9.2793,24.1368,108.5605,%97.12,84.41
1,Deney 2B - BART (Log Veri),6.4702,21.6842,109.2254,%89.42,27.68
2,Deney 3 - Bayesian Ridge (Matematiksel Hiyerarşi),2.0318,19.8132,109.9892,%89.42,27.88
